In [1]:
import secrets
import string
from ipywidgets import widgets
from IPython.display import display, HTML

# ----------------------------------------------------
# 1. CORE LOGIC (Python Backend)
# ----------------------------------------------------
def check_password_strength(pwd):
    """Evaluates password and returns score, feedback, and color."""
    if not pwd:
        return 0, "Enter a password...", "#333", []

    score = 0
    suggestions = []

    # Length Check
    if len(pwd) >= 8: score += 1
    else: suggestions.append("Make it at least 8 characters long.")
    if len(pwd) >= 12: score += 1 # Bonus

    # Complexity Checks
    if any(c.islower() for c in pwd): score += 1
    else: suggestions.append("Add lowercase letters.")

    if any(c.isupper() for c in pwd): score += 1
    else: suggestions.append("Add uppercase letters.")

    if any(c.isdigit() for c in pwd): score += 1
    else: suggestions.append("Add at least one number.")

    if any(c in string.punctuation for c in pwd): score += 1
    else: suggestions.append("Add special characters (e.g., !, @, #).")

    # Uniqueness Check
    unique_chars = len(set(pwd))
    if unique_chars < len(pwd) * 0.5:
        score = max(1, score - 1)
        suggestions.append("Avoid repetitive characters (e.g., 'aaaa').")

    # Determine UI Mapping based on Score
    if score <= 2:
        return 25, "Weak ❌", "#ff4d4d", suggestions
    elif score in (3, 4):
        return 50, "Fair ⚠️", "#ffa500", suggestions
    elif score == 5:
        return 75, "Good 👍", "#2ecc71", suggestions
    else:
        return 100, "Strong & Secure 💪", "#1abc9c", suggestions

def generate_alternatives():
    """Generates 2 secure random passwords."""
    pool = string.ascii_letters + string.digits + "!@#$%^&*()_+"
    alts = []
    for _ in range(2):
        alt = "".join(secrets.choice(pool) for _ in range(14))
        alts.append(alt)
    return alts

# ----------------------------------------------------
# 2. UI COMPONENTS & STYLING (IPython Widgets)
# ----------------------------------------------------
# Custom CSS for modern styling inside Colab
display(HTML("""
<style>
    .analyzer-card { font-family: 'Segoe UI', sans-serif; max-width: 400px; padding: 15px; border-radius: 8px; background: #ffffff; box-shadow: 0 4px 12px rgba(0,0,0,0.1); }
    .alt-box { background: #f0f4f8; padding: 6px 12px; border-radius: 4px; font-family: monospace; font-size: 1.05rem; margin-top: 5px; border: 1px solid #d0e1f9; display: inline-block; }
    .suggestion-item { color: #e74c3c; font-size: 0.9rem; margin-bottom: 3px; }
</style>
"""))

title = widgets.HTML("<h2>Password Strength Analyzer</h2><p style='color:#666;'>Type below to test strength</p>")
password_input = widgets.Password(placeholder='Type password...', description='Password:')
progress_bar = widgets.IntProgress(value=0, min=0, max=100, step=1, bar_style='', orientation='horizontal', layout=widgets.Layout(width='100%', height='12px'))
feedback_label = widgets.HTML(value="<b>Enter a password...</b>")
suggestions_box = widgets.VBox()
alternatives_box = widgets.VBox()

# Combine layout into a clean interface container
ui_container = widgets.VBox(
    [title, password_input, progress_bar, feedback_label, suggestions_box, alternatives_box],
    layout=widgets.Layout(padding='20px', border='1px solid #ddd', border_radius='8px', width='450px', background_color='#fff')
)

# ----------------------------------------------------
# 3. INTERACTIVE EVENT HANDLER
# ----------------------------------------------------
def on_value_change(change):
    pwd = change['new']
    percent, text, color, suggestions = check_password_strength(pwd)

    # Update progress bar and text color
    progress_bar.value = percent
    progress_bar.style.bar_color = color
    feedback_label.value = f"<span style='color: {color}; font-size: 1.1rem; font-weight: bold;'>{text}</span>"

    # Update suggestions display
    if suggestions and percent < 100:
        sug_html = "<h4>Suggestions to improve:</h4><ul>" + "".join(f"<li class='suggestion-item'>{s}</li>" for s in suggestions) + "</ul>"
        suggestions_box.children = [widgets.HTML(sug_html)]
    else:
        suggestions_box.children = []

    # Update alternatives display if password is not already strong
    if pwd and percent < 75:
        alts = generate_alternatives()
        alt_html = "<h4>Stronger Alternatives:</h4>" + "".join(f"<div class='alt-box'>{a}</div><br>" for a in alts)
        alternatives_box.children = [widgets.HTML(alt_html)]
    else:
        alternatives_box.children = []

# Watch the password field for typing changes
password_input.observe(on_value_change, names='value')

# Render everything
display(ui_container)